# Wind forecast explorer

Analyze forecast wind speed and direction, including a polar view that makes direction shifts and stronger periods easy to identify.

This live demonstration uses public meteo@uniparthenope data. Values are generated
automatically and are intended for research, education, and general information—not for official
warnings, navigation, or safety-critical decisions. Times are UTC. Run and edit the cells in this
browser-based workspace; your changes stay in your browser unless you download the notebook.


In [ ]:
import json
import os
import sys
from urllib.parse import urlencode
from datetime import datetime

if sys.platform == "emscripten":
    import piplite
    await piplite.install("ipywidgets")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

API_BASE = "https://api.meteo.uniparthenope.it" if sys.platform == "emscripten" else os.environ.get(
    "METEO_API_BASE_URL", "https://api.meteo.uniparthenope.it"
).rstrip("/")
BRAND = "#1f4e79"
ACCENT = "#2a91eb"

def api_json(path, params=None):
    url = f"{API_BASE}/{path.lstrip('/')}"
    if params:
        url = f"{url}?{urlencode(params)}"
    if sys.platform == "emscripten":
        from pyodide.http import open_url
        return json.load(open_url(url))
    import requests
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def forecast_frame(place):
    payload = api_json(f"products/wrf5/timeseries/{place}")
    if payload.get("result") != "ok":
        raise ValueError(f"No WRF forecast is available for {place}")
    frame = pd.DataFrame(payload["timeseries"])
    frame["time"] = pd.to_datetime(frame["dateTime"], format="%Y%m%dZ%H%M", utc=True)
    return frame.set_index("time")

PLACES = {"Gulf of Naples": "ca001", "Gulf of Salerno": "ca002", "Cilento": "ca003"}


In [ ]:
place = widgets.Dropdown(options=PLACES, description="Area:")
hours = widgets.IntSlider(value=48, min=12, max=168, step=12, description="Hours:")
output = widgets.Output()

def render(*_):
    with output:
        clear_output(wait=True)
        try:
            frame = forecast_frame(place.value).head(hours.value)
            theta = np.deg2rad(frame.wd10.astype(float))
            speed = frame.ws10.astype(float)
            fig = plt.figure(figsize=(9, 5.2))
            polar = fig.add_subplot(121, projection="polar")
            polar.set_theta_zero_location("N")
            polar.set_theta_direction(-1)
            points = polar.scatter(theta, speed, c=np.arange(len(frame)), cmap="viridis", s=35)
            polar.set_title("Direction and speed")
            timeline = fig.add_subplot(122)
            timeline.plot(frame.index, speed, color=BRAND, lw=2)
            timeline.set(ylabel="Wind speed (m/s)", title=place.label)
            timeline.grid(alpha=.25)
            fig.colorbar(points, ax=polar, pad=.12, label="Forecast step")
            fig.autofmt_xdate()
            plt.show()
        except Exception as exc:
            display(HTML(f"<b>Forecast request failed:</b> {exc}"))

place.observe(render, names="value")
hours.observe(render, names="value")
display(widgets.VBox([widgets.HBox([place, hours]), output]))
render()
